# Part B — real relationship pipeline on local Qwen3-VL-8B, GPU-only (Colab)

Local-model counterpart to the GPT-5.5-low run done earlier this session
(`src/relation_bench/arms/openai_call_llm.py` + `run_real_extraction_partB.py`, not in
this repo's git history — ad hoc scratch scripts). This notebook runs the SAME real,
unmodified `pnid-extraction-agent` relationship pipeline — `ocr_reasoning_extract` →
`build_result` (baseline `loop_member` relations) → `apply_hierarchy` (hierarchy pass +
chunked connectivity pass: `feeds`/`relieves_to`/`actuates`/`signal_to`) — with the ONLY
swap being `call_llm`: local Qwen3-VL-8B (`extraction_local.qwen_call_llm.build_qwen_call_llm`)
instead of GPT-5.5-low. This is exactly what Benchmark_Gaps_Register.md's gap #12 shim was
built to make possible — the real code, a swappable model, zero agent-source edits.

**Same 3 sheets as the GPT-5.5-low run**, for a direct comparison:
`PX-2368-0180004-001`, `GD-B-540-DP-2920-005-Z`, `PX-2365-0140006-001`.

**Same deviation as the GPT-5.5-low run, same reason**: prod's OCR step needs
`GOOGLE_CLOUD_VISION_API_KEY`, which isn't available. These are real, born-digital vector
PDFs, so PyMuPDF's own embedded text layer (`page.get_text("words")`) substitutes —
genuine, perfect text, not a compromise (prod's own `PNID_HYBRID_TOKENS` code already
treats vector PDF text as a legitimate token source for exactly this reason).

GPU/CPU split (per project convention): everything CPU-only (code install, sheet download,
vector-text extraction) happens before the model loads; the GPU is only used for the
3 real inference runs, then freed.

In [1]:
!nvidia-smi

Fri Jul 24 11:45:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Config

In [2]:
# ── Model config ────────────────────────────────────────────────────────
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
QWEN_MAX_NEW_TOKENS = 4096

# ── HF (private agent-code zip, sheet archives, results push). No Google Drive, ever. ──
import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/pnid-extraction-datasets"
EXTRACTION_AGENT_SRC_REPO = "timthy45/pnid-extraction-agent-src"
EXTRACTION_AGENT_SRC_FILE = "agent_src/latest.zip"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE", "paste your HF token"

# ── The 3 sheets — SAME ones the real GPT-5.5-low run used, for direct comparison. ──
# Paths match the real single-level zip structure (AG_PNID.zip -> "AG_PNID/<file>.pdf",
# RIVE_LTTS_Sample.zip -> "RIVE/<file>.pdf"), confirmed by the existing
# ExtractionAgent_Local_GPUOnly.ipynb notebook's own verified SHEETS paths — not the
# double-nested local-scratchpad layout this session's local exploration used.
AG_DIR = "/content/sheets/AG_PNID"
RIVE_DIR = "/content/sheets/RIVE"

SHEETS = [
    ("PX-2368-0180004-001", f"{RIVE_DIR}/PX-2368-0180004-001.pdf"),
    ("GD-B-540-DP-2920-005-Z", f"{AG_DIR}/GD-B-540-DP-2920-005-Z.pdf"),
    ("PX-2365-0140006-001", f"{RIVE_DIR}/PX-2365-0140006-001.PDF"),
]
SMOKE_STEM = "PX-2368-0180004-001"   # smallest/fastest of the 3, and the one with the
                                       # MBD-0100 finding already understood — run this first

## 2. Install (GPU-runtime prep) — private code

**Risk worth knowing about, not just hoping it's fine:** this pulls `agent_src/latest.zip`
as-is — if that zip was packaged before `extraction_local/qwen_generate.py` existed (or
before other modules this cell needs were added), the import check below will raise a
clear `RuntimeError` naming exactly what's missing, rather than failing silently later.
If that happens, repackage the zip locally (whatever script/process built `latest.zip`
previously) and re-push it, then re-run this cell.

In [3]:
import zipfile, sys
from pathlib import Path
from huggingface_hub import hf_hub_download

!pip install -q pymupdf   # pnid_pipeline.rasterize/triage/hierarchy import fitz directly

AGENT_SRC_ROOT = Path("/content/agent_src")
AGENT_SRC_ROOT.mkdir(exist_ok=True)

# force_download=True: this zip has been repackaged and re-pushed to the SAME
# agent_src/latest.zip path multiple times in quick succession while debugging the Qwen
# call_llm fixes (2026-07-23) -- confirmed live that a plain hf_hub_download() kept
# serving a stale cached copy even after re-running this cell (one fix's presence check
# kept failing repeatedly despite the source being correct locally and re-pushed).
# force_download bypasses any local/CDN cache entirely.
zp = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO, filename=EXTRACTION_AGENT_SRC_FILE,
                      repo_type="dataset", token=HF_TOKEN, force_download=True)
with zipfile.ZipFile(zp) as zf:
    zf.extractall(AGENT_SRC_ROOT)
print("extracted:", sorted(p.name for p in AGENT_SRC_ROOT.iterdir()))

AGENT_DIR = str(AGENT_SRC_ROOT / "agents" / "pnid-extraction-agent")
PID_ML_SRC = str(AGENT_SRC_ROOT / "pid_ml_src")
for p in (AGENT_DIR, PID_ML_SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

import importlib
REQUIRED_MODULES = [
    "pnid_pipeline.ocr_reasoning", "pnid_pipeline.hierarchy", "pnid_pipeline.assemble",
    "pnid_pipeline.triage", "pnid_pipeline.rasterize", "pnid_pipeline.run",
    "pnid_pipeline.models", "pnid_pipeline.llm_proxy",
    "extraction_local.qwen_generate", "extraction_local.qwen_call_llm",
]
missing = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f"{mod}: {type(e).__name__}: {e}")
if missing:
    raise RuntimeError("Missing/broken imports:\n  " + "\n  ".join(missing))
print(f"all {len(REQUIRED_MODULES)} required modules import cleanly")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 107.7 MB/s eta 0:00:0000:0100:01


agent_src/latest.zip: reconstructing file:   0%|          |  0.00B /  625kB            

agent_src/latest.zip: downloading bytes:           |  0.00B            

extracted: ['agents', 'pid_ml_src']
all 10 required modules import cleanly


## 3. Sheet PDFs — download

In [5]:
from huggingface_hub import hf_hub_download
import zipfile

for fname in ("AG_PNID.zip", "RIVE_LTTS_Sample.zip"):
    zp = hf_hub_download(repo_id=DATA_REPO, filename=f"sheets/{fname}",
                          repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall("/content/sheets")
    print(f"extracted {fname}")

from pathlib import Path
missing_sheets = [(stem, p) for stem, p in SHEETS if not Path(p).exists()]
if missing_sheets:
    raise FileNotFoundError(f"missing sheet PDFs: {missing_sheets}")
print(f"all {len(SHEETS)} sheet PDFs present.")

sheets/AG_PNID.zip: reconstructing file:   0%|          |  0.00B / 63.0MB            

sheets/AG_PNID.zip: downloading bytes:           |  0.00B            

extracted AG_PNID.zip


sheets/RIVE_LTTS_Sample.zip: reconstructing file:   0%|          |  0.00B / 20.3MB            

sheets/RIVE_LTTS_Sample.zip: downloading bytes:           |  0.00B            

extracted RIVE_LTTS_Sample.zip
all 3 sheet PDFs present.


## 4. Load Qwen3-VL-8B + build `call_llm`

In [4]:
import torch
from extraction_local.qwen_generate import load_qwen_model, build_qwen_generate_fn
from extraction_local.qwen_call_llm import build_qwen_call_llm

qwen_model, qwen_processor = load_qwen_model(QWEN_MODEL_ID)
print("Qwen3-VL-8B loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

qwen_generate_fn = build_qwen_generate_fn(qwen_model, qwen_processor,
                                           max_new_tokens_default=QWEN_MAX_NEW_TOKENS)

# max_new_tokens_cap: real measured decode speed on this A100 was ~17 tok/s (diagnosed
# live, 2026-07-23) -- honoring the real pipeline's own max_tokens=32000 per call would
# take 30+ minutes PER CALL, impractical for a multi-call-per-sheet run. This cap OVERRIDES
# that per-call request (max_new_tokens_cap always wins, by design -- see qwen_call_llm.py's
# docstring) at a budget that comfortably covers a dense sheet's real tag count (~150 tags
# at ~50-70 tokens/tag JSON entry) while bounding worst-case latency to ~8000/17 =~ 470s
# (~8 min) per call. Raise if a sheet's tag count needs more; lower if latency matters more
# than completeness for a given run.
QWEN_MAX_NEW_TOKENS_CAP = 8000
call_llm = build_qwen_call_llm(qwen_generate_fn, max_new_tokens_cap=QWEN_MAX_NEW_TOKENS_CAP)
print(f"call_llm ready (max_new_tokens_cap={QWEN_MAX_NEW_TOKENS_CAP}, same contract "
      "apply_hierarchy expects: prompt, schema_model, images=, model=, max_tokens=)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL-8B loaded. VRAM: 17.5 GB
call_llm ready (max_new_tokens_cap=8000, same contract apply_hierarchy expects: prompt, schema_model, images=, model=, max_tokens=)


### 4.1 Force-reload `qwen_call_llm` and rebuild `call_llm`

Re-running the install cell (section 2) only re-extracts files to disk — it does NOT
re-import already-imported Python modules in a live kernel, and `call_llm` above was built
from whatever `build_qwen_call_llm` was in memory at the time section 4 first ran. Run this
cell (no need to reload the 8B model itself) to guarantee the smoke gate below actually
uses the fixed code, then re-run the smoke gate cell.

In [32]:
QWEN_MAX_NEW_TOKENS_CAP = 8000

In [38]:
import importlib
import inspect
import extraction_local.qwen_call_llm as _qcl_mod
import extraction_local.qwen_generate as _qg_mod

importlib.reload(_qcl_mod)
importlib.reload(_qg_mod)

# Prove all fixes are actually the version now in memory, not just hope so.
_src = inspect.getsource(_qcl_mod.build_qwen_call_llm)
_has_max_tokens_fix = "elif max_tokens is not None" in _src
_src_example = inspect.getsource(_qcl_mod._schema_instruction)
_has_example_fix = "_example_instance" in _src_example
_src_generate = inspect.getsource(_qg_mod.build_qwen_generate_fn)
_has_repetition_penalty = "repetition_penalty" in _src_generate
# FIXED (2026-07-23): the old check ("no_repeat_ngram_size" not in _src_generate) was a
# false alarm generator -- it matched the EXPLANATORY COMMENT documenting why the
# parameter was removed, not just a live keyword argument. Check specifically for the
# live-argument form "no_repeat_ngram_size=" instead, which only appears if it's actually
# being passed to model.generate() again.
_has_ngram_removed = "no_repeat_ngram_size=" not in _src_generate
print(f"max_tokens fix present: {_has_max_tokens_fix}")
print(f"worked-example schema-instruction fix present: {_has_example_fix}")
print(f"repetition_penalty present: {_has_repetition_penalty}")
print(f"no_repeat_ngram_size correctly REMOVED (as a live argument): {_has_ngram_removed}")
if not (_has_max_tokens_fix and _has_example_fix and _has_repetition_penalty and _has_ngram_removed):
    raise RuntimeError(
        "The reloaded extraction_local modules don't match the corrected version. Re-check "
        "agent_src/latest.zip was actually repackaged after the no_repeat_ngram_size removal, "
        "or /content/agent_src has an old copy sys.path is finding first. Do not proceed "
        "until all four print True."
    )

qwen_generate_fn = _qg_mod.build_qwen_generate_fn(qwen_model, qwen_processor,
                                                  max_new_tokens_default=QWEN_MAX_NEW_TOKENS)
call_llm = _qcl_mod.build_qwen_call_llm(qwen_generate_fn,
                                        max_new_tokens_cap=QWEN_MAX_NEW_TOKENS_CAP)
print("qwen_generate_fn and call_llm rebuilt from the reloaded (corrected) modules.")

max_tokens fix present: True
worked-example schema-instruction fix present: True
repetition_penalty present: True
no_repeat_ngram_size correctly REMOVED (as a live argument): True
qwen_generate_fn and call_llm rebuilt from the reloaded (corrected) modules.


## 4.5. Diagnose a smoke-gate failure (0 tags, 0 relationships)

`build_qwen_call_llm`'s `call_llm` swallows JSON-parse failures and silently returns an
empty default instance (same "honest failure mode" convention as the real prod proxy) — so
"0 tags" doesn't tell you WHY. `generate_fn` itself is called OUTSIDE that try/except, so it
didn't raise (no traceback) — meaning it returned *some* string that failed to parse as the
expected JSON schema. This cell calls the raw `qwen_generate_fn` directly, bypassing the
schema layer entirely, so you can see exactly what the model actually produced. Run each
sub-cell in order — text-only first (confirms the model/processor loaded correctly at all),
then image+short-prompt (confirms image encoding works), then the real dense-chunk-shaped
prompt (confirms the real failure).

In [10]:
# --- Step 1: text-only, no image, no schema — does generation work AT ALL? ---
raw1 = qwen_generate_fn("Reply with exactly the word: pong", [])
print(f"[text-only] len={len(raw1)}  repr={raw1!r}")

[text-only] len=4  repr='pong'


In [11]:
# --- Step 2: image + short prompt — does image encoding/understanding work? ---
import base64
import fitz as _fitz

_doc = _fitz.open(dict(SHEETS)[SMOKE_STEM])
_pg = _doc[0]
_pix = _pg.get_pixmap(matrix=_fitz.Matrix(1.0, 1.0))
_b64 = base64.b64encode(_pix.tobytes("png")).decode()
print(f"test image: {_pix.width}x{_pix.height}")

raw2 = qwen_generate_fn("Describe what kind of technical drawing this is, in one sentence.", [_b64])
print(f"[image+short-prompt] len={len(raw2)}  repr={raw2!r}")

test image: 1224x792
[image+short-prompt] len=188  repr='This is a detailed piping and instrumentation diagram (P&ID) for a bulk oil storage and charging system, showing fluid flow paths, equipment, valves, instrumentation, and interconnections.'


In [12]:
# --- Step 3: the REAL prompt shape ocr_reasoning_extract actually builds — reusing its
# own internal _prompt/_split_regions/_overview_b64, not a hand-approximated copy, so this
# is a faithful reproduction of the real failing call. Self-contained (doesn't depend on
# section 5 running first) — duplicates a couple of small helpers on purpose. ---
import fitz as _fitz3
from pnid_pipeline import rasterize as _RZ3
from pnid_pipeline.triage import triage_page as _triage3
from pnid_pipeline.run import load_config as _load_config3, _load_env as _load_env3
from pnid_pipeline.ocr_reasoning import _prompt as _real_prompt, _split_regions, _overview_b64

_load_env3()
_CFG3 = _load_config3()


def _extract_vector_words3(pg, zoom):
    rmat = pg.rotation_matrix
    words = []
    for w in pg.get_text("words"):
        text = (w[4] or "").strip()
        if not text:
            continue
        r = (_fitz3.Rect(w[0], w[1], w[2], w[3]) * rmat) * zoom
        words.append((text, min(r.x0, r.x1), min(r.y0, r.y1), max(r.x0, r.x1), max(r.y0, r.y1)))
    return words


_smoke_pdf = dict(SHEETS)[SMOKE_STEM]
_doc3 = _fitz3.open(_smoke_pdf)
_pg3 = _doc3[0]
_tri3 = _triage3(_pg3, 0, _CFG3)
_zoom3 = _RZ3.work_zoom(_tri3.width_pt, _CFG3)
_img3, _W3, _H3 = _RZ3.render_page(_pg3, _zoom3)
_words3 = _extract_vector_words3(_pg3, _zoom3)
print(f"vector words extracted: {len(_words3)}")

_idx_words3 = [(i, t, (x0 + x1) / 2 / _W3, (y0 + y1) / 2 / _H3)
               for i, (t, x0, y0, x1, y1) in enumerate(_words3)]
_chunk_size = int(_CFG3.get("ocr_reasoning", {}).get("dense_chunk_tokens", 1500))
_regions3 = _split_regions(_idx_words3, _chunk_size)
print(f"regions (dense-sheet chunks): {len(_regions3)}, first region size: {len(_regions3[0])}")

_real_prompt_text = _real_prompt(_regions3[0], True)
print(f"real prompt length (chars): {len(_real_prompt_text)}")
_ov3 = _overview_b64(_img3)

raw3 = qwen_generate_fn(_real_prompt_text, [_ov3], max_new_tokens=4096)
print(f"\n[REAL prompt+image] raw output len={len(raw3)}")
print(f"--- first 2000 chars of raw output ---\n{raw3[:2000]}")

vector words extracted: 942
regions (dense-sheet chunks): 1, first region size: 942
real prompt length (chars): 22228

[REAL prompt+image] raw output len=11401
--- first 2000 chars of raw output ---
```json
{
  "standard": "P&ID",
  "area_prefix": "",
  "tags": [
    {
      "text": "MBD-0100",
      "type": "equipment",
      "word_ids": [
        "0"
      ]
    },
    {
      "text": "NBK-0300",
      "type": "equipment",
      "word_ids": [
        "1"
      ]
    },
    {
      "text": "XFMR—0301",
      "type": "equipment",
      "word_ids": [
        "2"
      ]
    },
    {
      "text": "BDV-0100A",
      "type": "equipment",
      "word_ids": [
        "3"
      ]
    },
    {
      "text": "BULK OIL SURGE VESSEL (BOSV)",
      "type": "equipment",
      "word_ids": [
        "4",
        "5",
        "6",
        "7",
        "8"
      ]
    },
    {
      "text": "BULK OIL TREATER",
      "type": "equipment",
      "word_ids": [
        "17",
        "18",
        "19"
    

In [13]:
# --- Step 4: did generation actually FINISH, or get cut off mid-JSON? Check the tail,
# and try the exact same parse path build_qwen_call_llm uses. ---
import json
from e2e_bench.backends.parse_json_common import _extract_json_text

print(f"--- last 300 chars of raw output ---\n{raw3[-300:]}")
print(f"\nends with closing fence: {raw3.rstrip().endswith('```')}")

candidate = _extract_json_text(raw3)
print(f"\n_extract_json_text found a candidate: {candidate is not None}")
if candidate is not None:
    print(f"candidate length: {len(candidate)}, last 200 chars: {candidate[-200:]!r}")
    try:
        parsed = json.loads(candidate)
        print(f"json.loads SUCCEEDED — {len(parsed.get('tags', []))} tags parsed")
    except Exception as e:
        print(f"json.loads FAILED: {type(e).__name__}: {e}")

--- last 300 chars of raw output ---
  ]
    },
    {
      "text": "PSV-0300C",
      "type": "safety_device",
      "word_ids": [
        "801"
      ]
    },
    {
      "text": "PSV-0300C",
      "type": "safety_device",
      "word_ids": [
        "802"
      ]
    },
    {
      "text": "PSV-0300C",
      "type": "safety_device",

ends with closing fence: False

_extract_json_text found a candidate: True
candidate length: 11328, last 200 chars: '-0300C",\n      "type": "safety_device",\n      "word_ids": [\n        "801"\n      ]\n    },\n    {\n      "text": "PSV-0300C",\n      "type": "safety_device",\n      "word_ids": [\n        "802"\n      ]\n    }'
json.loads FAILED: JSONDecodeError: Expecting ',' delimiter: line 705 column 6 (char 11328)


### 4.6 Go through the REAL `call_llm` path this time (schema instruction included)

Step 3 above bypassed `call_llm` entirely and called `qwen_generate_fn` directly — it never
got `_schema_instruction(OcrResult)` appended (the real prompt PLUS the full derived JSON
schema, which `call_llm` always adds). That's a real difference in input shape, not just a
shortcut. This cell wraps `generate_fn` to print exactly what it receives and returns when
invoked through the real, fixed `call_llm` — same call `ocr_reasoning_extract` itself makes
(`model=..., max_tokens=32000`), so whatever it shows is exactly what's happening in the
smoke gate above.

In [18]:
import time as _time4
from pnid_pipeline.ocr_reasoning import OcrResult as _OcrResult4


def _debug_generate_fn(prompt, images, max_new_tokens=None):
    print(f"[DEBUG] generate_fn called: prompt_len={len(prompt)}  n_images={len(images)}  "
          f"max_new_tokens={max_new_tokens!r}")
    t0 = _time4.monotonic()
    out = (qwen_generate_fn(prompt, images, max_new_tokens=max_new_tokens)
           if max_new_tokens is not None else qwen_generate_fn(prompt, images))
    dt = _time4.monotonic() - t0
    print(f"[DEBUG] generate_fn returned: elapsed={dt:.1f}s  output_len={len(out)}")
    print(f"[DEBUG] output (last 400 chars): {out[-400:]!r}")
    return out


debug_call_llm = _qcl_mod.build_qwen_call_llm(_debug_generate_fn)

result_direct = await debug_call_llm(
    _real_prompt_text, _OcrResult4, images=[_ov3],
    model="qwen3-vl-8b-local", max_tokens=32000,
)
print(f"\nparsed tags: {len(result_direct.tags)}")

[DEBUG] generate_fn called: prompt_len=23045  n_images=1  max_new_tokens=32000
[DEBUG] generate_fn returned: elapsed=3.8s  output_len=62
[DEBUG] output (last 400 chars): '```json\n{\n  "standard": "",\n  "prefix": "",\n  "tags": []\n}\n```'

parsed tags: 0


### 4.7 Isolate the real cause: schema-instruction text vs. a huge `max_new_tokens` value

Two things changed between the working manual test (step 3, real content) and the failing
real-path test (4.6, empty): the schema instruction got appended to the prompt, AND
`max_new_tokens` went from 4096 to 32000. Test each independently, same real prompt/image
both times.

In [19]:
# --- Test A: same real prompt (NO schema instruction), but max_new_tokens=32000 ---
# Isolates: does a huge max_new_tokens alone (no schema text) still work?
import time as _time5

t0 = _time5.monotonic()
rawA = qwen_generate_fn(_real_prompt_text, [_ov3], max_new_tokens=32000)
dtA = _time5.monotonic() - t0
print(f"[Test A: no schema instruction, max_new_tokens=32000] elapsed={dtA:.1f}s  len={len(rawA)}")
print(f"first 300 chars: {rawA[:300]!r}")

KeyboardInterrupt: 

In [20]:
# --- Test B: real prompt WITH schema instruction appended, but max_new_tokens=4096 ---
# Isolates: does adding the schema instruction text alone (small budget) still work?
from extraction_local.qwen_call_llm import _schema_instruction
from pnid_pipeline.ocr_reasoning import OcrResult as _OcrResult5

full_prompt_with_schema = _real_prompt_text + _schema_instruction(_OcrResult5)
print(f"prompt with schema instruction: {len(full_prompt_with_schema)} chars "
      f"(vs {len(_real_prompt_text)} without)")

t0 = _time5.monotonic()
rawB = qwen_generate_fn(full_prompt_with_schema, [_ov3], max_new_tokens=4096)
dtB = _time5.monotonic() - t0
print(f"\n[Test B: schema instruction included, max_new_tokens=4096] elapsed={dtB:.1f}s  len={len(rawB)}")
print(f"first 300 chars: {rawB[:300]!r}")

prompt with schema instruction: 23045 chars (vs 22228 without)

[Test B: schema instruction included, max_new_tokens=4096] elapsed=3.8s  len=62
first 300 chars: '```json\n{\n  "standard": "",\n  "prefix": "",\n  "tags": []\n}\n```'


### 4.8 Isolated cause confirmed: the schema-instruction TEXT triggers an empty response

Test B just proved it — schema instruction + the SAME 4096 budget that worked without it
→ still empty, in the same ~3.8s. Not a token-budget issue at all. GPT-5.5-low tolerates
this exact schema-dump style fine; Qwen3-VL-8B apparently doesn't — a real, specific model
weakness, not a code bug. Testing a concrete fix: replace the formal JSON-schema dump with
a short, concrete worked example instead (local/open-weight models are typically far more
reliable following a worked example than a machine-generated schema).

In [21]:
# --- Test C: replace the formal schema dump with a short worked example ---
_example_instruction = (
    "\n\nRespond with a SINGLE JSON object, no prose outside it, in exactly this shape "
    "(a worked example, not a template to copy literally):\n"
    '{"standard": "ISA-5.1", "prefix": "", "tags": ['
    '{"text": "FIC-101", "type": "instrument", "word_ids": [12], "bbox": []}, '
    '{"text": "P-100", "type": "equipment", "word_ids": [45, 46], "bbox": []}'
    "]}\n"
    "List EVERY real tag you found above as one entry in \"tags\" — do not stop early, "
    "do not return an empty list unless the drawing genuinely has no tags."
)

full_prompt_with_example = _real_prompt_text + _example_instruction
print(f"prompt with worked example: {len(full_prompt_with_example)} chars")

t0 = _time5.monotonic()
rawC = qwen_generate_fn(full_prompt_with_example, [_ov3], max_new_tokens=4096)
dtC = _time5.monotonic() - t0
print(f"\n[Test C: worked example instead of schema, max_new_tokens=4096] "
      f"elapsed={dtC:.1f}s  len={len(rawC)}")
print(f"first 500 chars: {rawC[:500]!r}")
print(f"last 300 chars: {rawC[-300:]!r}")

prompt with worked example: 22706 chars

[Test C: worked example instead of schema, max_new_tokens=4096] elapsed=234.8s  len=5317
first 500 chars: '{\n  "standard": "ISA-5.1",\n  "prefix": "",\n  "tags": [\n    {\n      "text": "MBD-0100",\n      "type": "equipment",\n      "word_ids": [0],\n      "bbox": []\n    },\n    {\n      "text": "NBK-0300",\n      "type": "equipment",\n      "word_ids": [1],\n      "bbox": []\n    },\n    {\n      "text": "XFMR-0301",\n      "type": "equipment",\n      "word_ids": [2],\n      "bbox": []\n    },\n    {\n      "text": "BDV-0100A",\n      "type": "equipment",\n      "word_ids": [3],\n      "bbox": []\n    },\n    {\n      "text":'
last 300 chars: ', 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763'


## 9. Probe 2 — Qwen verifier accuracy on real crops

**Why this probe exists**: the hybrid architecture proposal (geometry traces candidate
edges; Qwen only ever verifies via a short, bounded question on a crop — never open-ended
whole-sheet extraction) is only viable if Qwen is actually GOOD at that bounded task on
REAL AG/RIVE crops specifically. The ~80-84% connectivity yes/no figures cited earlier this
session came from PID2Graph crops, not this project's actual sheets — this probe checks
that number holds here, before any full build commits to the architecture.

**Data** (pre-built, hand-verified, before this Colab-side execution existed): 20 real
candidate symbol pairs across the 3 dev sheets (`PX-2368-0180004-001`,
`GD-B-540-DP-2920-005-Z`, `PX-2365-0140006-001`), spanning three source categories — pairs
the LLM and geometry tracer both agreed on, pairs only geometry traced, pairs only the LLM
claimed — each rendered as a crop with entity A boxed in RED and entity B boxed in BLUE
(250px margin, downscaled to max 900px), hand-verified by direct visual inspection against
the real drawing: 9 TRUE (really connected), 10 FALSE (not really connected), 1 SKIP (a real
bbox/tag-id lookup mismatch found during verification — the crop didn't show the expected
entities at all; excluded here as a data-quality issue, not a probe result, tracked
separately in `Benchmark_Gaps_Register.md`).

**Method**: download `probe_bundle.zip` (already pushed to `timthy45/pnid-extraction-datasets`
at `benchmarks/probe_bundle_2026-07-24.zip`), for each of the 19 valid (non-SKIP) pairs, ask
Qwen a single bounded yes/no question directly via `qwen_generate_fn` (not through
`call_llm`/schema layer — this is plain text-classification, not structured extraction), and
compute accuracy against the hand-verified `verdict` field. This is the exact bounded-task
shape the hybrid architecture would actually use in production (verify one candidate edge at
a time), not a proxy for it.

In [5]:
import json
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

PROBE_BUNDLE_FILE = "benchmarks/probe_bundle_2026-07-24.zip"
PROBE_BUNDLE_ROOT = Path("/content/probe_bundle")

_pb_zip = hf_hub_download(repo_id=DATA_REPO, filename=PROBE_BUNDLE_FILE,
                           repo_type="dataset", token=HF_TOKEN, force_download=True)
PROBE_BUNDLE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(_pb_zip) as zf:
    zf.extractall(PROBE_BUNDLE_ROOT)

with open(PROBE_BUNDLE_ROOT / "answer_key.json") as f:
    probe2_answer_key = json.load(f)
with open(PROBE_BUNDLE_ROOT / "probe3_answer_key.json") as f:
    probe3_answer_key = json.load(f)

print(f"probe bundle extracted: {sorted(p.name for p in PROBE_BUNDLE_ROOT.iterdir())}")
print(f"probe 2 candidates: {len(probe2_answer_key)} "
      f"(verdicts: {sorted(set(c['verdict'] for c in probe2_answer_key))})")
print(f"probe 3 candidates: {len(probe3_answer_key)}")

benchmarks/probe_bundle_2026-07-24.zip: reconstructing file:   0%|          |  0.00B / 1.76MB            

benchmarks/probe_bundle_2026-07-24.zip: downloading bytes:           |  0.00B            

probe bundle extracted: ['answer_key.json', 'candidates.json', 'pair_00.png', 'pair_01.png', 'pair_02.png', 'pair_03.png', 'pair_04.png', 'pair_05.png', 'pair_06.png', 'pair_07.png', 'pair_08.png', 'pair_09.png', 'pair_10.png', 'pair_11.png', 'pair_12.png', 'pair_13.png', 'pair_14.png', 'pair_15.png', 'pair_16.png', 'pair_17.png', 'pair_18.png', 'pair_19.png', 'probe3_answer_key.json', 'probe3_box_13.png', 'probe3_box_15.png', 'probe3_box_16.png', 'probe3_box_16B.png', 'probe3_box_17.png', 'probe3_box_22.png', 'probe3_box_22B.png', 'probe3_box_23.png']
probe 2 candidates: 20 (verdicts: ['FALSE', 'SKIP', 'TRUE'])
probe 3 candidates: 8


In [6]:
import base64

PROBE2_PROMPT = (
    "This image shows a crop of a P&ID drawing. Two symbols are highlighted with colored "
    "boxes: a RED box around one entity, and a BLUE box around a second entity. Is there a "
    "physical pipe or line directly connecting the RED-boxed entity to the BLUE-boxed "
    "entity? Answer with EXACTLY one word: YES or NO."
)


def _b64_of_png(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def _parse_yes_no(raw):
    t = raw.strip().upper()
    if t.startswith("YES"):
        return True
    if t.startswith("NO"):
        return False
    return None  # unparseable — counted as wrong below, not silently dropped


probe2_rows = []
for cand in probe2_answer_key:
    if cand["verdict"] == "SKIP":
        continue
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"] if "crop_file" in cand else \
        PROBE_BUNDLE_ROOT / f"pair_{cand['pair_id']:02d}.png"
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE2_PROMPT, [b64], max_new_tokens=16)
    pred = _parse_yes_no(raw)
    expected = cand["verdict"] == "TRUE"
    probe2_rows.append({
        "pair": img_path.name, "expected": cand["verdict"], "raw": raw.strip(),
        "pred": pred, "correct": (pred == expected),
    })
    print(f"{img_path.name}: expected={cand['verdict']:<5} raw={raw.strip()!r:<8} "
          f"correct={pred == expected}")

n_correct = sum(r["correct"] for r in probe2_rows)
n_total = len(probe2_rows)
n_unparseable = sum(r["pred"] is None for r in probe2_rows)
print(f"\nProbe 2 accuracy: {n_correct}/{n_total} = {n_correct / n_total:.1%}"
      f"  (unparseable responses: {n_unparseable})")

pair_00.png: expected=TRUE  raw='NO'     correct=False
pair_01.png: expected=TRUE  raw='NO'     correct=False
pair_02.png: expected=TRUE  raw='NO'     correct=False
pair_03.png: expected=FALSE raw='NO'     correct=True
pair_04.png: expected=TRUE  raw='NO'     correct=False
pair_05.png: expected=FALSE raw='NO'     correct=True
pair_06.png: expected=TRUE  raw='NO'     correct=False
pair_07.png: expected=FALSE raw='NO'     correct=True
pair_08.png: expected=FALSE raw='NO'     correct=True
pair_09.png: expected=FALSE raw='NO'     correct=True
pair_10.png: expected=TRUE  raw='NO'     correct=False
pair_12.png: expected=FALSE raw='NO'     correct=True
pair_13.png: expected=TRUE  raw='NO'     correct=False
pair_14.png: expected=TRUE  raw='NO'     correct=False
pair_15.png: expected=FALSE raw='NO'     correct=True
pair_16.png: expected=FALSE raw='NO'     correct=True
pair_17.png: expected=FALSE raw='NO'     correct=True
pair_18.png: expected=TRUE  raw='NO'     correct=False
pair_19.png: expect

## 10. Probe 3 — Qwen's accuracy reading real off-page connector annotations

**Why this probe exists**: off-page connector references (labeled border boxes with an
arrow and "FROM/TO [equipment] ([other sheet])" text) are a structural blind spot for any
single-sheet geometric tracer — confirmed this session as a real, recurring cause of
"agreement near-zero" measurements (most of GPT-5.5-low's disagreeing claims on
`PX-2368-0180004-001` turned out to reference off-page equipment, not tracer error). If the
hybrid architecture is going to close that gap by having Qwen read these connector boxes
directly (rather than leaving off-page claims entirely unverifiable), Qwen's raw
reading accuracy on REAL connector-box crops needs to be checked first.

**Data**: 8 real connector-box crops from `PX-2368-0180004-001`'s left border, each cropped
using EXACT `page.get_text("words")` coordinates for the referenced equipment string (not
visually estimated — an earlier visual-estimation attempt produced a wrong crop, see this
session's notes), 2 of 8 spot-checked as correct/clean. `probe3_answer_key.json` has each
crop's exact expected text, e.g. `"FROM GLYCOL CONDENSATE SEPARATOR (MBD-0635)"`.

**Method**: ask Qwen to transcribe the connector box's text verbatim, then check whether the
expected equipment tag (e.g. `MBD-0635`) appears in Qwen's transcription — a substring check
against the tag specifically, not exact full-string match, since the FROM/TO wording is
allowed to vary slightly as long as the referenced equipment is read correctly (that's the
actually load-bearing fact for reconnecting an off-page claim).

In [7]:
import re

PROBE3_PROMPT = (
    "This image shows an off-page connector box from the border of a P&ID drawing. "
    "Transcribe the text inside the box EXACTLY as written, verbatim, nothing else — no "
    "commentary, no extra words."
)


def _equipment_tag_from_expected(expected):
    m = re.search(r"\(([^)]+)\)\s*$", expected)
    return m.group(1) if m else expected


probe3_rows = []
for cand in probe3_answer_key:
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"]
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE3_PROMPT, [b64], max_new_tokens=64)
    tag = _equipment_tag_from_expected(cand["expected"])
    correct = tag.upper() in raw.upper()
    probe3_rows.append({
        "label": cand["label"], "expected": cand["expected"], "expected_tag": tag,
        "raw": raw.strip(), "correct": correct,
    })
    print(f"box {cand['label']}: expected_tag={tag!r:<14} raw={raw.strip()!r:<50} correct={correct}")

n_correct3 = sum(r["correct"] for r in probe3_rows)
n_total3 = len(probe3_rows)
print(f"\nProbe 3 accuracy: {n_correct3}/{n_total3} = {n_correct3 / n_total3:.1%}")

box 15: expected_tag='MBD-0635'     raw='15\n0180014-001 2"(150#)\nFROM GLYCOL\nCONDENSATE SEPARATOR\n(MBD-0635)\n\n22B\n0180005-001 2"(150#)' correct=True
box 22B: expected_tag='MBM-0400'     raw='22B\n0180005-001\n2"(150#)\nOILY REJECT FROM PRODU\nWATER HYDROCYCLONE\n(MBM-0400)' correct=True
box 17: expected_tag='PBA-0501/0502' raw='17\n0180012-002\n4"(150#)\nFROM LP FLARE SCRUBBE\n(PBA-0501/0502)' correct=True
box 16B: expected_tag='MBF-0920'     raw='16B\n0180006-001\n2"(150#)\nFROM VRU 2ND STAGE\nSUCTION SCRUBBER\n(MBF-0920)\n\n13\n16"(150#)' correct=True
box 13: expected_tag='MBD-4150'     raw='13\n0180003-001\n16"(150#)\nFROM LP DEGASSER\n(MBD-4150)' correct=True
box 16: expected_tag='PBA-0903/0953' raw='0180006-001 3"(150#) FROM VRU CONDENSATE (PBA-0903/0953)' correct=True
box 22: expected_tag='MBM-0400'     raw='26\n27\n28\n(PBA-0903/0953)\n22\n0180005-001\n3"(150#)\nFROM PRODUCED WATER\n(MBM-0400)' correct=True
box 23: expected_tag='HBG-0335'     raw='0180004-002'            

In [8]:
# --- Probes 2 & 3 combined verdict — the actual go/no-go gate for the hybrid architecture ---
PROBE_PASS_BAR = 0.80  # matches the ~80-84% PID2Graph figures this session cited as the bar to hold

p2_acc = n_correct / n_total if n_total else 0.0
p3_acc = n_correct3 / n_total3 if n_total3 else 0.0

print(f"Probe 1 (symbol-extent reconstruction from vector geometry): PASS (validated earlier, "
      f"including the harder branching case — see this notebook's/session's write-up)")
print(f"Probe 2 (connectivity verifier accuracy, real crops): {n_correct}/{n_total} = {p2_acc:.1%} "
      f"— {'PASS' if p2_acc >= PROBE_PASS_BAR else 'FAIL'} (bar: {PROBE_PASS_BAR:.0%})")
print(f"Probe 3 (off-page connector reading accuracy, real crops): {n_correct3}/{n_total3} = "
      f"{p3_acc:.1%} — {'PASS' if p3_acc >= PROBE_PASS_BAR else 'FAIL'} (bar: {PROBE_PASS_BAR:.0%})")

all_pass = p2_acc >= PROBE_PASS_BAR and p3_acc >= PROBE_PASS_BAR
print(f"\n{'ALL PROBES PASS' if all_pass else 'NOT ALL PROBES PASS'} — "
      f"{'proceed to the full hybrid build' if all_pass else 'do NOT proceed to a full build yet; report these numbers to Tom and revisit the architecture/bar before building anything further'}.")

Probe 1 (symbol-extent reconstruction from vector geometry): PASS (validated earlier, including the harder branching case — see this notebook's/session's write-up)
Probe 2 (connectivity verifier accuracy, real crops): 10/19 = 52.6% — FAIL (bar: 80%)
Probe 3 (off-page connector reading accuracy, real crops): 7/8 = 87.5% — PASS (bar: 80%)

NOT ALL PROBES PASS — do NOT proceed to a full build yet; report these numbers to Tom and revisit the architecture/bar before building anything further.


## 5. Vector-text extraction + the real relationship-pipeline runner

Mirrors the GPT-5.5-low run's `run_real_extraction_partB.py` exactly — same real
`ocr_reasoning_extract` → `build_result` → `apply_hierarchy` call sequence, same
vector-text-word substitution for the unavailable Vision OCR key, only `call_llm` differs.

In [7]:
import time
import fitz

from pnid_pipeline import rasterize as RZ
from pnid_pipeline.triage import triage_page
from pnid_pipeline.ocr_reasoning import ocr_reasoning_extract
from pnid_pipeline.hierarchy import apply_hierarchy
from pnid_pipeline.models import DrawingMeta
from pnid_pipeline.assemble import build_result
from pnid_pipeline.run import load_config, _load_env
from pnid_pipeline.llm_proxy import snapshot, delta

_load_env()
CFG = load_config()


def extract_vector_words(pg, zoom):
    """Real embedded PDF text layer, scaled to rendered-image pixel space — same
    substitution as the GPT-5.5-low run, same reason (no Vision OCR key here)."""
    rmat = pg.rotation_matrix
    words = []
    for w in pg.get_text("words"):
        text = (w[4] or "").strip()
        if not text:
            continue
        r = (fitz.Rect(w[0], w[1], w[2], w[3]) * rmat) * zoom
        words.append((text, min(r.x0, r.x1), min(r.y0, r.y1), max(r.x0, r.x1), max(r.y0, r.y1)))
    return words


async def run_sheet_qwen(pdf_path: str, model_name: str = "qwen3-vl-8b-local"):
    """Runs the REAL relationship pipeline (extraction + hierarchy + connectivity) on one
    sheet with the local Qwen call_llm. Returns (ExtractionResult, elapsed_s, n_calls)."""
    doc = fitz.open(pdf_path)
    pg = doc[0]
    tri = triage_page(pg, 0, CFG)
    zoom = RZ.work_zoom(tri.width_pt, CFG)
    img, W, H = RZ.render_page(pg, zoom)
    words = extract_vector_words(pg, zoom)

    before = snapshot(call_llm)
    t0 = time.monotonic()

    tags, ameta = await ocr_reasoning_extract(img, W, H, "", call_llm, model_name, CFG, words=words)
    meta = DrawingMeta(source_file=pdf_path.split("/")[-1], page=1,
                       sheet_size=tri.sheet_size, detected_standard=(ameta.get("standard") or None),
                       page_rotation_applied_deg=tri.rotation, render_dpi=int(72 * zoom),
                       canvas_px=(W, H), pdf_points=(tri.width_pt, tri.height_pt),
                       route="ocr_reasoning_vector_words")
    qa_extra = {"snap_rate": 0.0, "unclaimed_shapes": 0, "assertion_failures": [],
                "passes_run": ameta.get("passes", 1), "duration_seconds": 0.0, "cost_usd": 0.0}
    result = build_result(meta, tags, [], [], W, H, qa_extra)

    if CFG.get("hierarchy_pass", {}).get("enable", True):
        result = await apply_hierarchy(result, img, W, H, call_llm, model_name, CFG)

    elapsed = time.monotonic() - t0
    d = delta(before, snapshot(call_llm))
    n_calls = sum(u.get("calls", 0) for u in d.values())
    return result, elapsed, n_calls

## 6. Smoke gate — 1 sheet before spending GPU time on all 3

In [39]:
SMOKE_ONLY = True   # flip to False only after this gate passes and Tom says go

if SMOKE_ONLY:
    smoke_pdf = dict(SHEETS)[SMOKE_STEM]
    print(f"=== SMOKE: {SMOKE_STEM} ===")
    result, elapsed, n_calls = await run_sheet_qwen(smoke_pdf)
    n_tags = len(result.tags)
    n_rels = len(result.relationships)
    distinct_texts = len({t.text for t in result.tags})
    dup_ratio = 1 - (distinct_texts / n_tags) if n_tags else 0.0
    print(f"tags={n_tags}  relationships={n_rels}  llm_calls={n_calls}  elapsed={elapsed:.0f}s")
    print(f"distinct tag texts: {distinct_texts}/{n_tags}  (duplicate ratio: {dup_ratio:.1%})")
    print(f"Gate: at least one tag produced: {'PASS' if n_tags > 0 else 'FAIL'}")
    print(f"Gate: at least one relationship produced: {'PASS' if n_rels > 0 else 'FAIL'}")
    print(f"Gate: not a repetition loop (duplicate ratio < 50%): "
          f"{'PASS' if dup_ratio < 0.5 else 'FAIL'} ({dup_ratio:.1%})")
    print("Gate: wall-clock sane (compare to GPT-5.5-low's 153.5s on this same sheet — "
          f"local Qwen will likely be slower): {'note' if elapsed > 0 else 'FAIL'}, got {elapsed:.0f}s")
    print("sample tags:", [(t.id, t.text, t.type, t.parent_id) for t in result.tags[:10]])
    print("\nSTOP: review the gates above before setting SMOKE_ONLY=False.")

=== SMOKE: PX-2368-0180004-001 ===
tags=100  relationships=3  llm_calls=4  elapsed=472s
distinct tag texts: 12/100  (duplicate ratio: 88.0%)
Gate: at least one tag produced: PASS
Gate: at least one relationship produced: PASS
Gate: not a repetition loop (duplicate ratio < 50%): FAIL (88.0%)
Gate: wall-clock sane (compare to GPT-5.5-low's 153.5s on this same sheet — local Qwen will likely be slower): note, got 472s
sample tags: [('t0001', 'PSV-0300C', 'equipment', None), ('t0002', 'PSV-0300C', 'equipment', None), ('t0003', 'PSV-0300C', 'equipment', None), ('t0004', 'PSV-0300C', 'equipment', None), ('t0005', 'PSV-0300C', 'equipment', None), ('t0006', 'PSV-0300C', 'equipment', None), ('t0007', 'PSV-0300C', 'equipment', None), ('t0008', 'PSV-0300C', 'equipment', None), ('t0009', 'PSV-0300C', 'equipment', None), ('t0010', 'PSV-0300C', 'equipment', None)]

STOP: review the gates above before setting SMOKE_ONLY=False.


## 7. Full run — all 3 sheets (only after the smoke gate passes)

In [9]:
all_results = {}

if not SMOKE_ONLY:
    for stem, pdf_path in SHEETS:
        print(f"=== {stem} ===")
        result, elapsed, n_calls = await run_sheet_qwen(pdf_path)
        n_tags, n_rels = len(result.tags), len(result.relationships)
        print(f"  tags={n_tags}  relationships={n_rels}  llm_calls={n_calls}  elapsed={elapsed:.0f}s")
        all_results[stem] = {
            "dumped": result.model_dump(by_alias=True),
            "elapsed_s": elapsed, "llm_calls": n_calls,
            "n_tags": n_tags, "n_relationships": n_rels,
        }
    print("\ntotals:", {
        "tags": sum(r["n_tags"] for r in all_results.values()),
        "relationships": sum(r["n_relationships"] for r in all_results.values()),
        "llm_calls": sum(r["llm_calls"] for r in all_results.values()),
        "elapsed_s": sum(r["elapsed_s"] for r in all_results.values()),
    })
else:
    print("SMOKE_ONLY is still True — set it to False in the cell above and re-run that cell "
          "first, then re-run this one.")

SMOKE_ONLY is still True — set it to False in the cell above and re-run that cell first, then re-run this one.


## 8. Save + push results to HF, free GPU

In [ ]:
import json
from huggingface_hub import HfApi

if all_results:
    out_path = "/content/partb_qwen_relation_results.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=1)

    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo="benchmarks/partb_qwen_relation_results.json",
        repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
    print("results pushed to HF -> benchmarks/partb_qwen_relation_results.json")
else:
    print("no results to push yet — run section 7 first")

del qwen_model, qwen_processor
torch.cuda.empty_cache()

from google.colab import runtime
runtime.unassign()

In [9]:
for name in ["pair_00.png", "pair_04.png", "pair_13.png"]:
    b64 = _b64_of_png(PROBE_BUNDLE_ROOT / name)
    raw = qwen_generate_fn(
        "Describe the colored boxes you see in this image and what's between them.",
        [b64], max_new_tokens=100)
    print(name, "->", raw)
If it doesn't mention a red box and a blue box at all, the crops/overlay aren't reaching the model correctly (rendering, encoding, or resize issue) — that's a bug, not a capability finding.

2. Sanity-check the crops themselves — pull dims/size:
from PIL import Image
for name in ["pair_00.png", "pair_03.png"]:
    im = Image.open(PROBE_BUNDLE_ROOT / name)
    print(name, im.size, im.mode)

SyntaxError: invalid character '—' (U+2014) (4168367627.py, line 7)

In [10]:
for name in ["pair_00.png", "pair_04.png", "pair_13.png"]:
    b64 = _b64_of_png(PROBE_BUNDLE_ROOT / name)
    raw = qwen_generate_fn(
        "Describe the colored boxes you see in this image and what's between them.",
        [b64], max_new_tokens=100)
    print(name, "->", raw)

pair_00.png -> Based on the provided image, here is a description of the colored boxes and what lies between them:

1.  **Red Box:**
    *   **Label:** `MBD-010C`
    *   **Description:** This box highlights the name or identifier for the "BULK OIL SURGE VESSEL". It appears to be an equipment tag number.
    *   **What's Between Them?** There are no other colored boxes directly connected to it by piping
pair_04.png -> Based on the provided image, here is a description of the colored boxes and what lies between them:

1.  **The Colored Boxes:**
    *   There are two distinct colored rectangular boxes highlighting specific components.
    *   The box labeled `PSHL-0202B` (a pressure safety high limit valve) is outlined with a **blue** border.
    *   The label `PBA-0202` (identifying Pump B) is enclosed within
pair_13.png -> Based on the provided image, here is a description of the colored boxes and what lies between them:

1.  **The Colored Boxes:**
    *   There are two distinct colore

## 11. Probe 2b — CoT variant (does forcing a one-word answer suppress correct reasoning?)

Probe 2 (section 9) returned a constant `NO` on all 19 pairs — confirmed NOT a rendering/
encoding bug (diagnostic cells 39/40 show Qwen correctly reads both box labels and colors).
This is the last cheap check before concluding it's a genuine capability gap: let the model
reason/trace first, then extract a verdict from an `ANSWER:` line, instead of forcing
`max_new_tokens=16` and a bare YES/NO.

In [11]:
import re

PROBE2_COT_PROMPT = (
    "This image shows a crop of a P&ID drawing with a RED box around one entity and a "
    "BLUE box around a second entity. First, describe any pipe/line you can trace starting "
    "from the RED box. Then state your final answer as a new line: 'ANSWER: YES' or "
    "'ANSWER: NO'."
)


def _parse_cot_answer(raw):
    m = re.search(r"ANSWER:\s*(YES|NO)", raw, re.IGNORECASE)
    if not m:
        return None  # unparseable -- counted as wrong below, not silently dropped
    return m.group(1).upper() == "YES"


probe2_cot_rows = []
for cand in probe2_answer_key:
    if cand["verdict"] == "SKIP":
        continue
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"] if "crop_file" in cand else \
        PROBE_BUNDLE_ROOT / f"pair_{cand['pair_id']:02d}.png"
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE2_COT_PROMPT, [b64], max_new_tokens=200)
    pred = _parse_cot_answer(raw)
    expected = cand["verdict"] == "TRUE"
    probe2_cot_rows.append({
        "pair": img_path.name, "expected": cand["verdict"], "raw": raw.strip(),
        "pred": pred, "correct": (pred == expected),
    })
    print(f"{img_path.name}: expected={cand['verdict']:<5} pred={pred}  "
          f"correct={pred == expected}\n  reasoning: {raw.strip()[:200]!r}\n")

n_correct_cot = sum(r["correct"] for r in probe2_cot_rows)
n_total_cot = len(probe2_cot_rows)
n_unparseable_cot = sum(r["pred"] is None for r in probe2_cot_rows)
n_yes_cot = sum(r["pred"] is True for r in probe2_cot_rows)
print(f"\nProbe 2b (CoT) accuracy: {n_correct_cot}/{n_total_cot} = "
      f"{n_correct_cot / n_total_cot:.1%}  (unparseable: {n_unparseable_cot}, "
      f"said YES: {n_yes_cot}/{n_total_cot} -- if this is still 0, the flatline is real, "
      "not a one-word-forcing artifact)"
)

pair_00.png: expected=TRUE  pred=None  correct=False
  reasoning: 'Starting from the red-boxed entity labeled "MBD-010C BULK OIL SURGE VESSEL", I can trace the following:\n\n1. From MBD-010C (the surge vessel), there is an outlet pipe going to the right.\n2. This pipe b'

pair_01.png: expected=TRUE  pred=None  correct=False
  reasoning: 'Starting from the red-boxed entity "NBK-03QQ BULK OIL TREATERS", I can trace:\n\n1. A 16" x 14" pipe extending to the right, connecting to the bulk oil transformer (XFMR-0301).\n2. From that same connect'

pair_02.png: expected=TRUE  pred=None  correct=False
  reasoning: 'Starting from the red-boxed entity "MBD-010C BULK OIL SURGE VESSEL", I can trace:\n\n1. A 2" pipe (labeled “2””) extending downward from the vessel’s lower side.\n2. This 2" pipe connects to an FSV (Flow'

pair_03.png: expected=FALSE pred=True  correct=False
  reasoning: 'Starting from the red box labeled "HAM-010D", I can trace a 2" pipe that branches off to the right. This branch co

## 12. Probe 2c — CoT v2: fix truncation + force explicit endpoint check

Probe 2b (section 11) fixed the constant-NO flatline (6/19 said YES) but surfaced two new
issues, not yet separated: (1) 10/19 truncated mid-reasoning before ever reaching an
`ANSWER:` line at `max_new_tokens=200` -- purely mechanical; (2) of the 9 that DID finish,
5/9 = 55.6% correct -- roughly chance, and the reasoning traces *a* path from the red box
without ever explicitly confirming the endpoint is the blue-boxed entity specifically
(e.g. pair_04 stops at "a square symbol which repr[esents]..." with no check against BLUE).
This cell raises `max_new_tokens` to 350 and forces an explicit endpoint-match check before
the verdict, to isolate whether accuracy is really chance-level once truncation is fixed.

In [12]:
import re

PROBE2_COT_PROMPT_V2 = (
    "This image shows a crop of a P&ID drawing with a RED box around one entity and a BLUE "
    "box around a second entity. Trace any pipe/line starting from the RED box, step by "
    "step, briefly. Then explicitly state: does that traced line's endpoint match the "
    "BLUE-boxed entity specifically? Answer only after checking that. End with a new line: "
    "'ANSWER: YES' or 'ANSWER: NO'."
)


def _parse_cot_answer_v2(raw):
    m = re.search(r"ANSWER:\s*(YES|NO)", raw, re.IGNORECASE)
    if not m:
        return None  # unparseable (likely still truncated) -- counted as wrong, not dropped
    return m.group(1).upper() == "YES"


probe2_cot_v2_rows = []
for cand in probe2_answer_key:
    if cand["verdict"] == "SKIP":
        continue
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"] if "crop_file" in cand else \
        PROBE_BUNDLE_ROOT / f"pair_{cand['pair_id']:02d}.png"
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE2_COT_PROMPT_V2, [b64], max_new_tokens=350)
    pred = _parse_cot_answer_v2(raw)
    expected = cand["verdict"] == "TRUE"
    probe2_cot_v2_rows.append({
        "pair": img_path.name, "expected": cand["verdict"], "raw": raw.strip(),
        "pred": pred, "correct": (pred == expected),
    })
    print(f"{img_path.name}: expected={cand['verdict']:<5} pred={pred}  "
          f"correct={pred == expected}\n  reasoning: {raw.strip()[:250]!r}\n")

n_correct_v2 = sum(r["correct"] for r in probe2_cot_v2_rows)
n_total_v2 = len(probe2_cot_v2_rows)
n_unparseable_v2 = sum(r["pred"] is None for r in probe2_cot_v2_rows)
n_parseable_v2 = n_total_v2 - n_unparseable_v2
n_correct_parseable_v2 = sum(r["correct"] for r in probe2_cot_v2_rows if r["pred"] is not None)
print(f"\nProbe 2c (CoT v2) accuracy: {n_correct_v2}/{n_total_v2} = {n_correct_v2 / n_total_v2:.1%} "
      f"(unparseable/truncated: {n_unparseable_v2})")
if n_parseable_v2:
    print(f"Accuracy on PARSEABLE subset only: {n_correct_parseable_v2}/{n_parseable_v2} = "
          f"{n_correct_parseable_v2 / n_parseable_v2:.1%}  "
          "-- this is the number that answers 'is it really chance-level.'")

pair_00.png: expected=TRUE  pred=True  correct=True
  reasoning: 'The red box highlights "MBD-010C BULK OIL SURGE VESSEL". From this vessel, there is an outlet labeled “2” - 635 PSIG”, which connects to a valve (SDV 01000) then continues as a line labeled “2” - 620 PSIG”. This line leads directly into the blue boxe'

pair_01.png: expected=TRUE  pred=True  correct=True
  reasoning: 'The red box highlights "NBK-03QQ BULK OIL TREATERS". From this equipment, there is an outlet labeled “16”x14””, which connects via piping to the right side of the same vessel (or interface point). This 16"x14" line continues horizontally to the right'

pair_02.png: expected=TRUE  pred=True  correct=True
  reasoning: 'The red box highlights "MBD-010C BULK OIL SURGE VESSEL". From this vessel, there is an outlet labeled “6” - 245 PSIG”, which connects to a 2" x 1-1/2" line (with two check valves) leading directly to the blue boxed entity, “HAM-010”.\n\nYes, the traced'

pair_03.png: expected=FALSE pred=True  co

## 13. Probe 2d — same 19 pairs, but with the FINE-TUNED v3-relation adapter attached

Probes 2/2b/2c all ran on zero-shot BASE Qwen. But a dedicated LoRA adapter for exactly
this task (`v3-relation`, stage 12/10.5 connectivity) already exists at
`timthy45/qwen3vl-pnid-domain-base` -- it scored 89.2% vs GPT-5.5-low's 72.5-80% on
PID2Graph relation validation (only 1/3 training epochs done, per `E2E_Harness_Plan.md`).
It was dropped from the PID2Graph benchmark for contamination reasons (both PID2Graph
training trees are fully exhausted -- `Benchmark_Gaps_Register.md` gap #13). That
contamination reasoning does NOT apply here: this probe's crops come from this project's
real AG/RIVE sheets, which v3-relation was never trained on -- a genuinely clean test.
Attaches the adapter as a named PEFT adapter on the ALREADY-LOADED `qwen_model` (same
pattern as `PerStageV3_Stage13_Relation_vs_GPT55.ipynb` section 5) -- no reload of the
8B base needed, `qwen_generate_fn` keeps working unchanged since it holds a reference to
the same underlying module PEFT injects LoRA layers into in-place.

In [14]:
!pip uninstall -y torchao -q   # peft's LoRA dispatcher probes torchao and errors on an
                                 # incompatible pinned version even though quantization
                                 # isn't used here -- same fix PerStageV3_Stage13_Relation_
                                 # vs_GPT55.ipynb already applies. Uninstalling makes
                                 # is_torchao_available() return False instead of raising.
!pip install -q peft

from pathlib import Path
from peft import PeftModel
from huggingface_hub import snapshot_download

CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
RELATION_ADAPTER_PATH = "v3-relation/latest"  # whatever checkpoint exists now -- verify below

_relation_local = Path("/content/v3_relation_adapter")
snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                   allow_patterns=[f"{RELATION_ADAPTER_PATH}/*"], local_dir=str(_relation_local))
_relation_dir = _relation_local / RELATION_ADAPTER_PATH
assert (_relation_dir / "adapter_model.safetensors").exists(), f"missing: {_relation_dir}"
print(f"v3-relation checkpoint found: {_relation_dir}")

qwen_peft_model = PeftModel.from_pretrained(qwen_model, str(_relation_dir), adapter_name="relation")
qwen_peft_model.set_adapter("relation")
print("v3-relation adapter attached and active on qwen_model (in-place -- qwen_generate_fn "
      "still works unchanged)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

v3-relation checkpoint found: /content/v3_relation_adapter/v3-relation/latest
v3-relation adapter attached and active on qwen_model (in-place -- qwen_generate_fn still works unchanged)


In [15]:
# --- Probe 2d: rerun the CoT v2 prompt (section 12), v3-relation adapter active ---
probe2_relation_rows = []
for cand in probe2_answer_key:
    if cand["verdict"] == "SKIP":
        continue
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"] if "crop_file" in cand else \
        PROBE_BUNDLE_ROOT / f"pair_{cand['pair_id']:02d}.png"
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE2_COT_PROMPT_V2, [b64], max_new_tokens=350)
    pred = _parse_cot_answer_v2(raw)
    expected = cand["verdict"] == "TRUE"
    probe2_relation_rows.append({
        "pair": img_path.name, "expected": cand["verdict"], "raw": raw.strip(),
        "pred": pred, "correct": (pred == expected),
    })
    print(f"{img_path.name}: expected={cand['verdict']:<5} pred={pred}  "
          f"correct={pred == expected}\n  reasoning: {raw.strip()[:250]!r}\n")

n_correct_rel = sum(r["correct"] for r in probe2_relation_rows)
n_total_rel = len(probe2_relation_rows)
n_unparseable_rel = sum(r["pred"] is None for r in probe2_relation_rows)
n_parseable_rel = n_total_rel - n_unparseable_rel
n_correct_parseable_rel = sum(r["correct"] for r in probe2_relation_rows if r["pred"] is not None)
print(f"\nProbe 2d (v3-relation adapter, CoT v2 prompt) accuracy: "
      f"{n_correct_rel}/{n_total_rel} = {n_correct_rel / n_total_rel:.1%} "
      f"(unparseable/truncated: {n_unparseable_rel})")
if n_parseable_rel:
    print(f"Accuracy on PARSEABLE subset only: {n_correct_parseable_rel}/{n_parseable_rel} = "
          f"{n_correct_parseable_rel / n_parseable_rel:.1%}")
print(f"\nBASELINE for comparison -- base Qwen, same prompt, same crops (section 12): "
      f"9/17 = 52.9% on the parseable subset")
print("\nNOTE: v3-relation was frozen at only 1/3 training epochs (paused mid-epoch-0, "
      "never resumed per E2E_Harness_Plan.md) -- a real result but from an INCOMPLETE "
      "checkpoint. Interpret accordingly before making a final call on the architecture.")

pair_00.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_01.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_02.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_03.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_04.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_05.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_06.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_07.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_08.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_09.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_10.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_12.png: expected=FALSE pred=None  correct=False
  reasoning: 'No.'

pair_13.png: expected=TRUE  pred=None  correct=False
  reasoning: 'No.'

pair_14.png: expected=TRUE  pred=None  correct=Fals

## 14. Probe 2e — v3-relation adapter, ORIGINAL short-form prompt (not CoT)

Probe 2d's CoT prompt got a constant `'No.'` from the adapter -- almost certainly a
prompt-format mismatch, not a capability finding: `v3-relation` was trained/evaluated on
a short, forced `yes`/`no` prompt using bracketed pixel-coordinate references (see
`PerStageV3_Stage13_Relation_vs_GPT55.ipynb` cell 8's `build_relation_pool`), NOT a
paragraph-reasoning prompt. A long out-of-distribution prompt collapsing a narrow LoRA to
a fixed low-entropy default matches the known v2-adapter overfitting pattern. This reruns
the ORIGINAL Probe 2 prompt (section 9's `PROBE2_PROMPT`, `max_new_tokens=16`, forced
one-word YES/NO) -- much closer to the adapter's actual training distribution -- with
`v3-relation` still active (no reattach needed).

In [ ]:

# --- Probe 2e: same 19 pairs, ORIGINAL short-form Probe 2 prompt/parser, v3-relation active ---
probe2_relation_short_rows = []
for cand in probe2_answer_key:
    if cand["verdict"] == "SKIP":
        continue
    img_path = PROBE_BUNDLE_ROOT / cand["crop_file"] if "crop_file" in cand else \
        PROBE_BUNDLE_ROOT / f"pair_{cand['pair_id']:02d}.png"
    b64 = _b64_of_png(img_path)
    raw = qwen_generate_fn(PROBE2_PROMPT, [b64], max_new_tokens=16)
    pred = _parse_yes_no(raw)
    expected = cand["verdict"] == "TRUE"
    probe2_relation_short_rows.append({
        "pair": img_path.name, "expected": cand["verdict"], "raw": raw.strip(),
        "pred": pred, "correct": (pred == expected),
    })
    print(f"{img_path.name}: expected={cand['verdict']:<5} raw={raw.strip()!r:<8} "
          f"correct={pred == expected}")

n_correct_short = sum(r["correct"] for r in probe2_relation_short_rows)
n_total_short = len(probe2_relation_short_rows)
n_unparseable_short = sum(r["pred"] is None for r in probe2_relation_short_rows)
n_yes_short = sum(r["pred"] is True for r in probe2_relation_short_rows)
print(f"\nProbe 2e (v3-relation, short-form prompt) accuracy: "
      f"{n_correct_short}/{n_total_short} = {n_correct_short / n_total_short:.1%} "
      f"(unparseable: {n_unparseable_short}, said YES: {n_yes_short}/{n_total_short})")
print("\nReference points: base Qwen constant-NO (section 9) = 52.6%; "
      "base Qwen CoT v2 (section 12, parseable subset) = 52.9%; "
      "v3-relation CoT (section 13) = 0.0% (format mismatch, discard).")

pair_00.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_01.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_02.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_03.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_04.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_05.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_06.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_07.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_08.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_09.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_10.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_12.png: expected=FALSE raw='No.No.No.No.No.No.No.No.' correct=True
pair_13.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.' correct=False
pair_14.png: expected=TRUE  raw='No.No.No.No.No.No.No.No.